# Fit Enclosed mass and concentration

In [ ]:
%load_ext autoreload
%autoreload 2
import sys

sys.path.append("/pbs/home/m/maguena/git_codes/ClusterMassLike/")

In [ ]:
# from dsigma.helpers import dsigma_table
import astropy.units as u
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from astropy.cosmology import Planck15

In [ ]:
from clmm.cosmology.ccl import CCLCosmology

In [ ]:
cosmo = CCLCosmology(
    H0=Planck15.H0.value,
    Omega_dm0=Planck15.Om0 - Planck15.Ob0,
    Omega_b0=Planck15.Ob0,
    Omega_k0=Planck15.Ok0,
)

In [ ]:
import magic_mass

In [ ]:
m_bins = np.array([12.2, 12.5, 12.8, 13.1, 13.4, 13.7, 14.7])

In [ ]:
cmap = plt.get_cmap("magma")
norm = mpl.colors.BoundaryNorm(m_bins, cmap.N)
colors = [cmap(i / 6) for i in range(7)]

## Read data

* here is the list of arrays that we want and their shapes
(14, 15)["stacked_Dsigma"]
(14, 15)["stacked_theta"]
(14, 15)["stacked_variance"]
(400, 14, 15)["bs_theta"]
(400, 14, 15)["bs_Dsigma"]
['bs_Dsigma', 'bs_theta', 'stacked_Dsigma', 'stacked_theta', 'stacked_variance']

In [ ]:
# load boostrap table
bs_table = magic_mass.tools.read_ds_data(
    "/sps/lsst/users/lbaumont/software/dsigma/tutorial/old//bs_hsc_%d.parquet",
    m_bins,
    format="parquet",
)

In [ ]:
%%time
# load regular table
full_table = magic_mass.tools.read_ds_data(
    "/sps/lsst/users/lbaumont/software/dsigma/tutorial/old/boost_hsc_%d.csv",
    m_bins,
)
full_table["ds_t"].T.shape

## Measure mass

In [ ]:
mm = magic_mass.mass.MagicMass()

In [ ]:
mm.add_emass_table(full_table)

In [ ]:
mm.add_emass_table(bs_table, bootstrapped=True)

In [ ]:
full_table.emass["magic_mass_bs_mean"] = np.nanmean(
    bs_table.emass["magic_mass"], axis=-1
)
full_table.emass["magic_mass_bs_std"] = np.nanstd(bs_table.emass["magic_mass"], axis=-1)

In [ ]:
full_table.emass["gt_bs_mean"] = np.nanmean(bs_table.emass["gt"], axis=-1)
full_table.emass["gt_bs_std"] = np.nanstd(bs_table.emass["gt"], axis=-1)

## Compute M & R $\Delta$

Then $R_\Delta$ can be obtained by interpolating $\Delta(R)$, and $M_\Delta$ can also be estimated:

### Compute $\Delta(R)$

Using the encompassed masses, estimate $\Delta$ for each radius with:

$$
\Delta(R) = \frac{M(R)}{\frac{4\pi}{3} R^3 \rho_{\rm bkg}(z)}
$$

In [ ]:
bs_table.emass["Delatcrit_bkg"] = np.array(
    [
        magic_mass.delta.get_delta(mass, mm.radius, rho_bkg).value
        for mass, rho_bkg in zip(
            bs_table.emass["magic_mass"].transpose(2, 0, 1),
            Planck15.critical_density(bs_table["z_l"])
            .to(u.solMass / u.Mpc**3)
            .transpose(2, 0, 1),
        )
    ]
).transpose(1, 2, 0)

In [ ]:
# Delta critital bkg density
full_table.emass["Delatcrit_bkg"] = magic_mass.delta.get_delta(
    full_table.emass["magic_mass_bs_mean"],
    mm.radius,
    Planck15.critical_density(full_table["z_l"]).to(u.solMass / u.Mpc**3),
).value

### Compute M & R

Then $R_\Delta$ can be obtained by interpolating $\Delta(R)$, and $M_\Delta$ can also be estimated:

#### Compute

In [ ]:
%%time
for Delta in (500, 200):
    magic_mass.delta.add_mrdelta_to_bs_table(bs_table.emass, Delta)

In [ ]:
for Delta in (500, 200):
    magic_mass.delta.add_mrdelta_to_full_table(full_table.emass, Delta)

In [ ]:
for Delta in (500, 200):
    magic_mass.delta.add_mrdelta_err_to_full_table(full_table.emass, Delta)

## MCMC Fit mass and concentration

In [ ]:
pf = magic_mass.nfw.ProfileFit()

### Compute fit

In [ ]:
p0 = np.random.normal([13, 4], [0.1, 0.1], (32, 2))

In [ ]:
mcmc_fit_500 = magic_mass.mcmc_fit.MCMCFit(
    full_table, delta=500, fit_func=pf.func2h, cosmo=cosmo, nwalkers=32, clim=(0.1, 100)
)

In [ ]:
%%time
mcmc_fit_500.run_mcmc(p0=np.random.normal([13, 4], [0.1, 0.1], (32, 2)), nchain=10)

In [ ]:
for i in range(6):
    magic_mass.plot.plot_like(mcmc_fit_500.get_vstack_res()[i, :, 2000:])
    magic_mass.plot.plot_chain(mcmc_fit_500.get_vstack_res()[i, :, 2000:])

In [ ]:
# np.save("delta_500.npy", mcmc_fit_500.get_vstack_res())

In [ ]:
fig, axes = magic_mass.plot.plot_profiles_base(full_table, m_bins, colors)
magic_mass.plot.add_profile_mcmc(
    axes,
    full_table,
    mcmc_fit_500.fit,
    burnin=2000,
    func=lambda *args: pf.func2h(*args, delta=500, cosmo=cosmo),
    colors=colors,
    # ls=":",
)

In [ ]:
fig, ax = magic_mass.plot.plot_cm_base(m_bins, delta=200)

magic_mass.plot.add_mc_fit(
    ax,
    mcmc_fit_500.fit,
    2000,
    colors=colors,
    ls="",
    marker="^",
    label="Mass & conc. fit",
    lw=1,
    markersize=10,
    markeredgewidth=0.5,
    markerfacecolor="none",
)

label = "Enclosed mass"
for i in range(6):
    ax.axvline(
        full_table.emass["M500_crit"][i], color=colors[i], ls="--", lw=0.7, label=label
    )
    label = None


ax.set_xlim(4e11, 2e14)
ax.set_ylim(0.1, 200)
ax.legend()

### Compute fit with mass prior

In [ ]:
mcmc_fit_500_prior = magic_mass.mcmc_fit.MCMCFit(
    full_table,
    delta=500,
    fit_func=pf.func2h,
    cosmo=cosmo,
    nwalkers=32,
    clim=(0.1, 100),
    use_prior=True,
)

In [ ]:
%%time
mcmc_fit_500_prior.run_mcmc(
    p0=np.random.normal([13, 4], [0.1, 0.1], (32, 2)), nchain=10
)

In [ ]:
for i in range(6):
    magic_mass.plot.plot_like(mcmc_fit_500_prior.get_vstack_res()[i, :, 2000:])
    magic_mass.plot.plot_chain(mcmc_fit_500_prior.get_vstack_res()[i, :, 2000:])

In [ ]:
# np.save("delta_500_mprior.npy", mcmc_fit_500_prior.get_vstack_res())

In [ ]:
fig, axes = magic_mass.plot.plot_profiles_base(full_table, m_bins, colors)
magic_mass.plot.add_profile_mcmc(
    axes,
    full_table,
    mcmc_fit_500_prior.fit,
    burnin=2000,
    func=lambda *args: pf.func2h(*args, delta=500, cosmo=cosmo),
    colors=colors,
    # ls=":",
)

In [ ]:
fig, ax = magic_mass.plot.plot_cm_base(m_bins, delta=200)

magic_mass.plot.add_mc_fit(
    ax,
    mcmc_fit_500.fit,
    2000,
    colors=colors,
    ls="",
    marker="^",
    label="Mass & conc. fit",
    lw=1,
    markersize=10,
    markeredgewidth=0.5,
    markerfacecolor="none",
)
magic_mass.plot.add_mc_fit(
    ax,
    mcmc_fit_500_prior.fit,
    2000,
    colors=colors,
    ls="",
    marker="v",
    label="Mass & conc. fit (Mass prior)",
    lw=1,
    markersize=10,
    markeredgewidth=0.5,
    markerfacecolor="none",
)

label = "Enclosed mass"
for i in range(6):
    ax.axvline(
        full_table.emass["M500_crit"][i], color=colors[i], ls="--", lw=0.7, label=label
    )
    label = None


ax.set_xlim(4e11, 2e14)
ax.set_ylim(0.1, 200)
ax.legend()